# Autoresearch Experiment Analysis

Analysis of autonomous CIFAR-10 experiment results from `results.tsv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load the TSV (tab-separated, 5 columns: commit, val_acc, memory_gb, status, description)
df = pd.read_csv("results.tsv", sep="\t")
df["val_acc"] = pd.to_numeric(df["val_acc"], errors="coerce")
df["memory_gb"] = pd.to_numeric(df["memory_gb"], errors="coerce")
df["status"] = df["status"].str.strip().str.upper()

print(f"Total experiments: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
counts = df["status"].value_counts()
print("Experiment outcomes:")
print(counts.to_string())

n_keep = counts.get("KEEP", 0)
n_discard = counts.get("DISCARD", 0)
n_crash = counts.get("CRASH", 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")

In [ ]:
# Show all KEPT experiments (the improvements that stuck)
kept = df[df["status"] == "KEEP"].copy()
print(f"KEPT experiments ({len(kept)} total):\n")
for i, row in kept.iterrows():
    acc = row["val_acc"]
    desc = row["description"]
    print(f"  #{i:3d}  acc={acc:.6f}  mem={row['memory_gb']:.1f}GB  {desc}")

## Validation Accuracy Over Time

Track how the best kept `val_acc` evolves as experiments progress. The running maximum shows the frontier: the best result achieved so far.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

valid = df[df["status"] != "CRASH"].copy().reset_index(drop=True)
if valid.empty:
    raise ValueError("No non-crash experiments found in results.tsv")

baseline_acc = valid.loc[0, "val_acc"]
best_acc = valid["val_acc"].max()

# Plot discarded as faint background dots
disc = valid[valid["status"] == "DISCARD"]
ax.scatter(
    disc.index,
    disc["val_acc"],
    c="#cccccc",
    s=12,
    alpha=0.5,
    zorder=2,
    label="Discarded",
)

# Plot kept experiments as prominent green dots
kept_v = valid[valid["status"] == "KEEP"]
if kept_v.empty:
    raise ValueError("Need at least one KEEP row to plot progress")
ax.scatter(
    kept_v.index,
    kept_v["val_acc"],
    c="#2ecc71",
    s=50,
    zorder=4,
    label="Kept",
    edgecolors="black",
    linewidths=0.5,
)

# Running maximum step line
kept_mask = valid["status"] == "KEEP"
kept_idx = valid.index[kept_mask]
kept_acc = valid.loc[kept_mask, "val_acc"]
running_max = kept_acc.cummax()
ax.step(
    kept_idx,
    running_max,
    where="post",
    color="#27ae60",
    linewidth=2,
    alpha=0.7,
    zorder=3,
    label="Running best",
)

# Label each kept experiment with its description
for idx, acc in zip(kept_idx, kept_acc):
    desc = str(valid.loc[idx, "description"]).strip()
    if len(desc) > 45:
        desc = desc[:42] + "..."

    ax.annotate(
        desc,
        (idx, acc),
        textcoords="offset points",
        xytext=(6, 6),
        fontsize=8.0,
        color="#1a7a3a",
        alpha=0.9,
        rotation=20,
        ha="left",
        va="bottom",
    )

n_total = len(df)
n_kept = len(df[df["status"] == "KEEP"])
ax.set_xlabel("Experiment #", fontsize=12)
ax.set_ylabel("Validation Accuracy (higher is better)", fontsize=12)
ax.set_title(f"Autoresearch Progress: {n_total} Experiments, {n_kept} Kept Improvements", fontsize=14)
ax.legend(loc="lower right", fontsize=9)
ax.grid(True, alpha=0.2)

margin = max((best_acc - baseline_acc) * 0.15, 0.002)
ax.set_ylim(max(0.0, baseline_acc - margin), min(1.0, best_acc + margin))

plt.tight_layout()
plt.savefig("progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to progress.png")

## Summary Statistics

In [ ]:
# Summary stats
kept = df[df["status"] == "KEEP"].copy()
baseline_acc = df.iloc[0]["val_acc"]
best_acc = kept["val_acc"].max()
best_row = kept.loc[kept["val_acc"].idxmax()]

print(f"Baseline val_acc:  {baseline_acc:.6f}")
print(f"Best val_acc:      {best_acc:.6f}")
print(f"Total improvement: {best_acc - baseline_acc:+.6f} ({(best_acc - baseline_acc) * 100:.2f} percentage points)")
print(f"Best experiment:   {best_row['description']}")
print()

# How many experiments to find each improvement
print("Cumulative effort per improvement:")
kept_sorted = kept.reset_index()
for _, row in kept_sorted.iterrows():
    desc = str(row["description"]).strip()
    print(f"  Experiment #{row['index']:3d}: acc={row['val_acc']:.6f}  {desc}")

## Top Hits (Kept Experiments by Accuracy Gain)

In [ ]:
# Each kept experiment's delta is measured vs the previous kept experiment's accuracy
# (since experiments are cumulative -- each one builds on the last kept state)
kept = df[df["status"] == "KEEP"].copy()
kept["prev_acc"] = kept["val_acc"].shift(1)
kept["delta"] = kept["val_acc"] - kept["prev_acc"]

# Drop baseline (no delta)
hits = kept.iloc[1:].copy()

# Sort by delta improvement (biggest first)
hits = hits.sort_values("delta", ascending=False)

print(f"{'Rank':>4}  {'Delta':>8}  {'Acc':>10}  Description")
print("-" * 80)
for rank, (_, row) in enumerate(hits.iterrows(), 1):
    print(f"{rank:4d}  {row['delta']:+.6f}  {row['val_acc']:.6f}  {row['description']}")

print(f"\n{'':>4}  {hits['delta'].sum():+.6f}  {'':>10}  TOTAL improvement over baseline")